In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import pickle
import os
from PIL import Image
from matplotlib.pyplot import GridSpec
import snntorch as snn
from snntorch import spikegen
from snntorch import surrogate
from snntorch import spikeplot
from sklearn.preprocessing import LabelEncoder
from snntorch import utils

In [ ]:
DATA_DIR = "../../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [ ]:
# data = pd.read_parquet(f"{DATASET_DIR}/csecicids2018.parquet")

In [ ]:
# label_mapping = {
#     'Benign': 'Benign',
#     'Bot': 'Botnet',
#     'FTP-BruteForce': 'Brute Force',
#     'SSH-Bruteforce': 'Brute Force',
#     'DDoS attacks-LOIC-HTTP': 'DDoS',
#     'DDOS attack-LOIC-UDP': 'DDoS',
#     'DDOS attack-HOIC': 'DDoS',
#     'DoS attacks-GoldenEye': 'DoS',
#     'DoS attacks-Slowloris': 'DoS',
#     'DoS attacks-SlowHTTPTest': 'DoS',
#     'DoS attacks-Hulk': 'DoS',
#     'Infilteration': 'Infiltration',
#     'Brute Force -Web': 'Brute Force',
#     'Brute Force -XSS': 'Brute Force',
#     'SQL Injection': 'Infiltration'  # Assuming SQL Injection is part of Infiltration
# }

# data["Label"] = data["Label"].map(label_mapping)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            return img, label

In [ ]:
train_df = pd.read_csv(f"{DATASET_DIR}/train1_multi.csv")
test_df = pd.read_csv(f"{DATASET_DIR}/test1_multi.csv")
val_df = pd.read_csv(f"{DATASET_DIR}/val1_multi.csv")

In [ ]:
LE = LabelEncoder()
LE.fit(train_df["Label"].unique())

# LE.classes_

# swap classes in the label encoder
# swapped_classes = LE.classes_.copy()
# swapped_classes[0], swapped_classes[1] = swapped_classes[1], swapped_classes[0]

# LE.classes_ = swapped_classes

train_df_encoded = train_df.copy()
train_df_encoded["Label"] = LE.transform(train_df_encoded["Label"])

val_df_encoded = val_df.copy()
val_df_encoded["Label"] = LE.transform(val_df_encoded["Label"])

train_df_encoded.head()

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

train_dataset = CustomDataset(train_df_encoded, f"{DATASET_DIR}/images", transform=transform)
val_dataset = CustomDataset(val_df_encoded, f"{DATASET_DIR}/images", transform=transform)

In [ ]:
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight

labels = train_df_encoded["Label"].values
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [ ]:
BATCH_SIZE = 40
train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class SNNClassifier(nn.Module):
    def __init__(self, num_classes=2, time_steps=4, threshold=0.3, alpha=0.3):
        super(SNNClassifier, self).__init__()
        self.time_steps = time_steps
        
        # Spiking layers
        self.conv1 = ConvSpikingLayer(1, 16, 4, 4, threshold, alpha)
        self.conv2 = ConvSpikingLayer(16, 32, 2, 2, threshold, alpha)
        
        # Time-value encoder
        self.encoder = TimeValEncoder(time_steps)
        
        # Classifier
        self.fc = nn.Linear(32*4*4, num_classes)
        
        # State trackers
        self.spk1 = self.spk2 = None
        self.mem1 = self.mem2 = None

    def forward(self, x):
        # Add time dimension: (B,C,H,W) → (T,B,C,H,W)
        x = x.unsqueeze(0).repeat(self.time_steps, 1, 1, 1, 1)
        
        # Process through layers
        spk1, mem1 = self.conv1(x)
        spk2, mem2 = self.conv2(spk1)
        
        # Temporal encoding
        encoded = self.encoder(spk2)


        self.spk1 = spk1
        self.spk2 = spk2
        self.mem1 = mem1
        self.mem2 = mem2
        
        out = self.fc(encoded.flatten(1))
        return torch.softmax(out, dim=1)

class ConvSpikingLayer(nn.Module):
    """Single convolutional spiking layer with document-specific reset"""
    def __init__(self, in_channels, out_channels, kernel_size, stride, threshold=0.3, alpha=0.3):
        super(ConvSpikingLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              stride, padding='valid')
        
        self.lif = snn.Leaky(
            beta=1.0,  # No leakage: V(t) = V(t-1) + I(t)
            threshold=threshold,
            reset_mechanism="none",  # Disable built-in reset
            spike_grad=surrogate.fast_sigmoid(),
            output=True
        )
        self.alpha = alpha

    def forward(self, x):
        """Input shape: (T, B, C, H, W)"""
        time_steps, batch_size = x.shape[:2]
        spk_rec = []
        mem_rec = []

        mem = self.lif.reset_mem()
        
        for t in range(time_steps):
            conv_out = self.conv(x[t])
            spk, mem = self.lif(conv_out, mem)
            
            # Document-specific reset: (V - V_thr) * α
            mem = torch.where(spk > 0,
                             (mem - self.lif.threshold) * self.alpha,
                             mem)
            
            spk_rec.append(spk)
            mem_rec.append(mem)
            
        return torch.stack(spk_rec, dim=0), torch.stack(mem_rec, dim=0)

class TimeValEncoder(nn.Module):
    def __init__(self, time_steps):
        super(TimeValEncoder, self).__init__()
        weights = [2**(time_steps-i-1) for i in range(time_steps)]
        weights = torch.tensor(weights, dtype=torch.float32)
        self.weights = nn.Parameter(weights/weights.sum(), requires_grad=False)

    def forward(self, x):
        # x: (time_steps, batch_size, channels, height, width)
        return torch.einsum('tb...,t->b...', x, self.weights.to(x.device))
    
class CustomLoss(nn.Module):
    def __init__(self, num_classes):
        super(CustomLoss, self).__init__()
        self.n_classes = num_classes

    def forward(self, predict, target):
        # Convert targets to one-hot encoding
        target_onehot = torch.zeros_like(predict).scatter(1, target.unsqueeze(1), 1)
        
        # 1. Compute α term (difference between max prediction and correct class score)
        cor = (predict * target_onehot).sum(dim=1)  # Correct class scores
        pre = predict.max(dim=1)[0]                 # Max prediction scores
        alpha = pre - cor

        # 2. Compute β term (ranking penalty)
        val = predict.gather(1, target.unsqueeze(1)).squeeze()  # Correct class values
        ids = (predict > val.unsqueeze(1)).sum(dim=1).float()   # Number of classes ranked higher
        beta = 1 - cor

        # 3. Final loss (Eq. in Algorithm 2)
        loss = (self.n_classes * alpha + (ids + 1) * beta).mean()
        return loss    
    

class CubicSplineThreshold:
    """Implements the document's cubic spline-based threshold logic"""
    def __init__(self, scaling_factor=10):
        self.scaling_factor = scaling_factor  # Controls threshold sensitivity

    def compute_threshold(self, mem_rec):
        """
        mem_rec: (T, B, C, H, W) membrane potentials
        Returns: Gradient clipping threshold using slope approximation
        """
        if mem_rec is None:
            return 1.0  # Fallback
        
        # Simplified cubic spline slope approximation (doc Eq. 9-10)
        mem_sample = mem_rec[:, 0].detach().flatten().cpu().numpy()  # First sample
        x = np.arange(len(mem_sample))
        
        # Cubic spline fitting (doc Eq. 7-8)
        from scipy.interpolate import CubicSpline
        cs = CubicSpline(x, mem_sample)
        derivatives = cs(x, 1)  # First derivative (slope)
        
        avg_slope = np.mean(np.abs(derivatives))
        return max(0.1, min(5.0, avg_slope * self.scaling_factor))

In [ ]:
dummy = torch.randn(40, 1, 32, 32)
model = SNNClassifier(num_classes=6, time_steps=4)
outputs = model(dummy)

spk1 = model.spk1
spk2 = model.spk2

spk1.shape, spk2.shape, outputs.shape

1024

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def train_model(train_dataloader, val_dataloader, num_epochs=50, lr=0.001, weight_decay=0, num_classes=6, model_savepath=None, device='cuda', dt_ms=1.0):
    model = SNNClassifier(num_classes=num_classes, time_steps=4).to(device)
    
    # Initialize optimizer and scheduler
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda t: 1.0 if t==0 else 0.5 * (1 + np.cos(((t + 1) / num_epochs) * np.pi)))

    loss_fn = CustomLoss(num_classes=num_classes)
    threshold_fn = CubicSplineThreshold(scaling_factor=15)

    history = {
    'train_loss': [],
    'val_loss': [],
    'avg_train_loss': [],
    'avg_val_loss': [],
    'train_acc': [],
    'val_acc': [],
    'spike_stats': {
        'train_avg_spikes_per_neuron': [],
        'train_spike_rate_hz': [],
        'val_avg_spikes_per_neuron': [],
        'val_spike_rate_hz': [],
        'train_active_neurons_percent': [],
        'val_active_neurons_percent': [],
        'membrane_potential_avg': [],
        'membrane_potential_std': [],
        'threshold_proximity_avg': [],
        'threshold_proximity_std': []
    },
    'best_val_loss': float('inf'),
    'best_model_state': None
    }

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # --- TRAINING PHASE ---
        model.train()
        epoch_train_loss = 0
        spike_stats = {
            'total_spikes': 0,
            'total_neurons': 0,
            'active_neurons': 0,
            'membrane_sum': 0.0,
            'membrane_sq_sum': 0.0,
            'proximity_sum': 0.0
        }

        for data, targets in tqdm(train_dataloader, desc="Training"):
            data, targets = data.to(device), targets.to(device)
            utils.reset(model)
            
            # Forward pass
            outputs = model(data)
            spk1, spk2 = model.spk1, model.spk2
            
            # Calculate dynamic thresholds
            thresh_conv1 = threshold_fn.compute_threshold(model.mem1)
            thresh_conv2 = threshold_fn.compute_threshold(model.mem2)

            # Loss calc
            loss = loss_fn(outputs, targets)
            epoch_train_loss += loss.item()

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping with dynamic thresholds
            torch.nn.utils.clip_grad_norm_(model.conv1.parameters(), thresh_conv1)
            torch.nn.utils.clip_grad_norm_(model.conv2.parameters(), thresh_conv2)
            torch.nn.utils.clip_grad_norm_(model.fc.parameters(), 1.0)
            
            optimizer.step()

            # --- SPIKE STATISTICS ---
            with torch.no_grad():
                # Spike counts
                spike_stats['total_spikes'] += (spk1.sum() + spk2.sum()).item()
                
                # Neuron counts
                batch_size = data.size(0)
                spike_stats['total_neurons'] += (spk1[0].numel() + spk2[0].numel()) * batch_size
                
                # Active neurons
                spike_stats['active_neurons'] += (
                    (spk1.sum(dim=0) > 0).sum().item() + 
                    (spk2.sum(dim=0) > 0).sum().item()
                )
                
                # Membrane statistics
                if model.mem1 is not None and model.mem2 is not None:
                    mem = torch.cat([model.mem1.flatten(), model.mem2.flatten()])
                    spike_stats['membrane_sum'] += mem.sum().item()
                    spike_stats['membrane_sq_sum'] += (mem**2).sum().item()
                    
                    # Threshold proximity (assuming 0.3 threshold from document)
                    proximity = torch.abs(mem - 0.3)
                    spike_stats['proximity_sum'] += proximity.sum().item()

        # --- EPOCH STATISTICS ---
        avg_train_loss = epoch_train_loss / len(train_dataloader)
        history['avg_train_loss'].append(avg_train_loss)
        
        # Spike metrics
        avg_spikes = spike_stats['total_spikes'] / spike_stats['total_neurons'] if spike_stats['total_neurons'] > 0 else 0
        active_percent = (spike_stats['active_neurons'] / spike_stats['total_neurons']) * 100 if spike_stats['total_neurons'] > 0 else 0
        
        # Membrane metrics
        membrane_avg = spike_stats['membrane_sum'] / spike_stats['total_neurons'] if spike_stats['total_neurons'] > 0 else 0
        membrane_std = np.sqrt(
            (spike_stats['membrane_sq_sum'] / spike_stats['total_neurons']) - membrane_avg**2
        ) if spike_stats['total_neurons'] > 0 else 0
        
        # Threshold proximity
        proximity_avg = spike_stats['proximity_sum'] / spike_stats['total_neurons'] if spike_stats['total_neurons'] > 0 else 0

        # Update history
        history['spike_stats']['train_avg_spikes_per_neuron'].append(avg_spikes)
        history['spike_stats']['train_spike_rate_hz'].append(avg_spikes * (1000/dt_ms))
        history['spike_stats']['train_active_neurons_percent'].append(active_percent)
        history['spike_stats']['membrane_potential_avg'].append(membrane_avg)
        history['spike_stats']['membrane_potential_std'].append(membrane_std)
        history['spike_stats']['threshold_proximity_avg'].append(proximity_avg)

        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Spike Rate: {avg_spikes*(1000/dt_ms):.1f}Hz | Active Neurons: {active_percent:.1f}%")

        # --- VALIDATION PHASE ---
        with torch.no_grad():
            model.eval()
            val_loss = 0
            correct = 0
            total = 0
            val_spike_stats = {
                'total_spikes': 0,
                'total_neurons': 0,
                'active_neurons': 0,
                'membrane_sum': 0.0,
                'proximity_sum': 0.0
            }
            
            for data, targets in tqdm(val_dataloader, desc="Validation"):
                data, targets = data.to(device), targets.to(device)
                utils.reset(model)
                
                outputs = model(data)
                spk1, spk2 = model.spk1, model.spk2
                
                # Loss and accuracy
                loss = loss_fn(outputs, targets)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == targets).sum().item()
                total += targets.size(0)
                
                # Validation statistics
                val_spike_stats['total_spikes'] += (spk1.sum() + spk2.sum()).item()
                val_spike_stats['total_neurons'] += (spk1[0].numel() + spk2[0].numel()) * data.size(0)
                val_spike_stats['active_neurons'] += (
                    (spk1.sum(dim=0) > 0).sum().item() + 
                    (spk2.sum(dim=0) > 0).sum().item()
                )
                
                if model.mem1 is not None and model.mem2 is not None:
                    mem = torch.cat([model.mem1.flatten(), model.mem2.flatten()])
                    val_spike_stats['membrane_sum'] += mem.sum().item()
                    proximity = torch.abs(mem - 0.3)
                    val_spike_stats['proximity_sum'] += proximity.sum().item()

        # Validation metrics
        avg_val_loss = val_loss / len(val_dataloader)
        val_acc = correct / total
        history['avg_val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        # Update validation spike stats
        val_avg_spikes = val_spike_stats['total_spikes'] / val_spike_stats['total_neurons'] if val_spike_stats['total_neurons'] > 0 else 0
        val_active_percent = (val_spike_stats['active_neurons'] / val_spike_stats['total_neurons']) * 100 if val_spike_stats['total_neurons'] > 0 else 0
        val_membrane_avg = val_spike_stats['membrane_sum'] / val_spike_stats['total_neurons'] if val_spike_stats['total_neurons'] > 0 else 0
        val_proximity_avg = val_spike_stats['proximity_sum'] / val_spike_stats['total_neurons'] if val_spike_stats['total_neurons'] > 0 else 0

        history['spike_stats']['val_avg_spikes_per_neuron'].append(val_avg_spikes)
        history['spike_stats']['val_spike_rate_hz'].append(val_avg_spikes * (1000/dt_ms))
        history['spike_stats']['val_active_neurons_percent'].append(val_active_percent)
        history['spike_stats']['threshold_proximity_std'].append(val_proximity_avg)

        # Model checkpointing
        if avg_val_loss < history['best_val_loss']:
            history['best_val_loss'] = avg_val_loss
            history['best_model_state'] = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_loss': avg_val_loss,
                'val_acc': val_acc
            }
            print(f"New best validation loss: {avg_val_loss:.4f}")

        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2%}")
        print(f"Val Spike Rate: {val_avg_spikes*(1000/dt_ms):.4f}Hz | Active Neurons: {val_active_percent:.4f}%")
        print("-" * 50)

        # Update learning rate
        scheduler.step()

    # Final model saving
    if model_savepath:
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history
        }, model_savepath)
        
        if history['best_model_state']:
            best_path = model_savepath.replace(".pt", "_best.pt")
            torch.save(history['best_model_state'], best_path)

    return model, history
    

In [ ]:
base_dir = "../../models/checkpoints/paper_multisnn"
path = f"{base_dir}/modelv1.pt"

os.makedirs(base_dir, exist_ok=True)
model_savepath = path

In [ ]:
model, history = train_model(train_data_loader, val_data_loader, num_epochs=50, lr=0.001, weight_decay=0, num_classes=6, model_savepath=model_savepath, device=device, dt_ms=1.0)

In [ ]:
test_df_encoded = test_df.copy()
test_df_encoded["Label"] = LE.transform(test_df_encoded["Label"])

test_dataset = CustomDataset(test_df_encoded, f"{DATASET_DIR}/images", transform=val_transform)
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
history.keys()

In [ ]:
def plot_training_results(history):
    """Plot training results including loss curves and spike statistics."""
    plt.figure(figsize=(18, 15))

    # plot loss curves
    plt.subplot(3, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='red')
    plt.title('Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # plot accuracy curves
    plt.subplot(3, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc', color='blue')
    plt.plot(history['val_acc'], label='Val Acc', color='red')
    plt.title('Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # plot average spikes per neuron
    plt.subplot(3, 2, 3)
    plt.plot(history['train_avg_spikes_per_neuron'], label='Train', color='blue')
    plt.plot(history['val_avg_spikes_per_neuron'], label='Validation', color='red')
    plt.title('Average Spikes per Neuron')
    plt.xlabel('Epoch')
    plt.ylabel('Avg. Spikes per Neuron')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_results(history)

In [ ]:
# sample 12000 data points from test data with equal distribution of benign and malicious samples
test_data_sample = test_df_encoded.groupby("Label").sample(2000, random_state=42)

test_dataset_sample = CustomDataset(test_data_sample, f"{DATASET_DIR}/images", transform=val_transform)
test_data_loader_sample = DataLoader(test_dataset_sample, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def test_model(model, test_dataloader, device="cuda"):
    with torch.no_grad():
        model.eval()
        correct = 0
        total = 0
        total_spikes = 0
        total_neurons = None
        all_preds = []
        all_labels = []
        
        for inputs, labels in tqdm(test_dataloader, desc=f'Testing'):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            
            # Collect test spikes
            spikes_conv1 = torch.sum(model.spk1).item()
            spikes_conv2 = torch.sum(model.spk2).item()
            total_spikes += spikes_conv1 + spikes_conv2

            # Calculate number of spiking neurons
            if total_neurons is None:
                conv1_neurons = model.conv1.conv.out_channels * model.spk1.shape[2:].numel()
                conv2_neurons = model.conv2.conv.out_channels * model.spk2.shape[2:].numel()
                total_neurons = conv1_neurons + conv2_neurons

            predicted = torch.argmax(outputs.data, 1)
            total += labels.size(0)

            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        accuracy = correct / total
        avg_spikes = total_spikes / (total_neurons * model.time_steps * len(test_dataloader.dataset))

        print(f'Test Accuracy: {accuracy:.4f} | Avg. Spikes per Neuron: {avg_spikes:.4f}')

    return accuracy, all_preds, all_labels

acc, y_pred, y_true = test_model(model, test_data_loader_sample, device=device)

In [ ]:
def plot_cm(y_true, y_pred, classes, title='Confusion Matrix'):
    """Plot confusion matrix."""
    from sklearn.metrics import confusion_matrix, classification_report
    import seaborn as sns
    import matplotlib.pyplot as plt

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


    class_report = classification_report(y_true, y_pred, target_names=classes)
    print(class_report)

plot_cm(y_true, y_pred, LE.classes_, title='Confusion Matrix')